# 🧠 Home NIDS — Train Model (ตัวเทรนหลัก)

Notebook นี้คือที่เทรนโมเดลหลักของโปรเจกต์ รันจากบนลงล่างได้เลย (**Kernel → Restart & Run All**)

| ขั้น | ทำอะไร |
|---|---|
| 0 | ตั้งค่า (ดึงจาก `config/config.yaml` + โค้ดของ microservice) |
| 1 | แปลง `.pcap` → CSV (`training/pcap/normal/`, `training/pcap/test/`) |
| 2 | โหลดข้อมูลเทรน (traffic ปกติ) |
| 3 | ดูการกระจายของ feature |
| 4 | เทรน StandardScaler + Isolation Forest |
| 5 | สร้างชุดทดสอบที่มี label (ปกติ / โจมตี) |
| 6 | เลือก threshold |
| 7 | วัดผล: **Confusion Matrix (TP/FP/TN/FN), Accuracy, Precision, Recall, F1, FPR, ROC-AUC** |
| 8 | บันทึกโมเดลลง `system/models/` → detection-service ใช้ต่อทันที |

> **วิธีเพิ่ม dataset:** วาง `.pcap` traffic ปกติใน `training/pcap/normal/`, ไฟล์ที่มีการโจมตีใน `training/pcap/test/`
> แล้วเขียนกฎ label (IP ผู้โจมตี + ช่วงเวลา) ใน `training/pcap/labels.csv`
> ถ้ายังไม่มีไฟล์ notebook ยังรันได้ด้วยข้อมูลจำลอง

## 0. ตั้งค่า
ค่าทั้งหมดมาจาก `config/config.yaml` (ไฟล์เดียวกับที่ microservice ใช้) แก้ใน config หรือ override ในเซลล์นี้ได้

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "system"))   # โค้ดของระบบ: shared/, services/

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib import font_manager
_fonts = {f.name for f in font_manager.fontManager.ttflist}            # ใช้ฟอนต์ไทยเท่าที่มีในเครื่อง
plt.rcParams["font.family"] = [f for f in ("Tahoma", "Leelawadee UI", "Noto Sans Thai", "Loma") if f in _fonts] or ["DejaVu Sans"]

# ---- โค้ดของ microservice (ใช้ตัวเดียวกับตอนรันจริง) ----
from shared.config import load_config, resolve_path
from shared.features import FEATURE_NAMES, FEATURE_VERSION        # สูตร feature ของ feature-service
# ---- โค้ดฝั่ง training ----
from training.generate_data import generate, load_packets_csv
from training.pcap_to_csv import convert_folder, list_pcaps          # ใช้ packet_to_meta ของ capture-service
from training.common import (build_features, train_model, anomaly_scores, recommend_threshold,
                             make_scenarios, is_ping_scenario, save_artifacts)
from training import evaluate as ev

cfg = load_config()
T = cfg["training"]

# ================= PARAMETERS (แก้ได้) =================
WINDOW_SEC     = cfg["feature"]["window_sec"]     # ต้องเท่ากับ feature-service
N_ESTIMATORS   = T["n_estimators"]
SEED           = T["seed"]
FP_QUANTILE    = T["fp_quantile"]
MAX_PACKETS    = T["max_packets_per_pcap"]        # อ่านไม่เกินกี่ packet ต่อไฟล์ pcap
NORMAL_DIR     = resolve_path(T["pcap_normal_dir"])
TEST_DIR       = resolve_path(T["pcap_test_dir"])
LABELS_FILE    = resolve_path(T["labels_file"])
PCAP_CSV_DIR   = resolve_path(T["pcap_csv_dir"])

USE_SYNTHETIC   = True    # รวมข้อมูลปกติจำลอง (แนะนำ: ทำให้โมเดลรู้จัก ICMP ที่ dataset อาจไม่มี)
DROP_ICMP_ECHO  = True    # ตัด ping ออกจากข้อมูลเทรน (ไม่งั้นโมเดลจะเรียนว่า ping ปกติ)
USE_SIM_TEST    = True    # เพิ่มชุดทดสอบจำลอง (ping + nmap) เข้าไปในการวัดผล
FORCE_RECONVERT = False   # True = แปลง pcap ใหม่แม้มี CSV แล้ว
THRESHOLD_MODE  = "recommended"   # "recommended" | "best_f1" | หรือใส่ตัวเลข เช่น 0.63
# =======================================================
print("config:", resolve_path("config/config.yaml"))
print(f"window={WINDOW_SEC}s  n_estimators={N_ESTIMATORS}  features={len(FEATURE_NAMES)} (v{FEATURE_VERSION})")

## 1. แปลง pcap → CSV
อ่านไฟล์ใน `training/pcap/normal/` และ `training/pcap/test/` แปลงด้วยฟังก์ชันเดียวกับ capture-service เก็บไว้ที่ `data/pcap_csv/` (ครั้งถัดไปใช้ของเดิม ไม่ต้องแปลงซ้ำ)

In [ ]:
print("pcap traffic ปกติ :", [p.name for p in list_pcaps(NORMAL_DIR)] or "ไม่มี")
print("pcap สำหรับทดสอบ :", [p.name for p in list_pcaps(TEST_DIR)] or "ไม่มี")
t0 = time.time()
normal_csvs = convert_folder(NORMAL_DIR, PCAP_CSV_DIR, MAX_PACKETS, FORCE_RECONVERT)
test_csvs   = convert_folder(TEST_DIR,   PCAP_CSV_DIR, MAX_PACKETS, FORCE_RECONVERT)
print(f"เสร็จใน {time.time() - t0:.1f}s")

## 2. โหลดข้อมูลเทรน (traffic ปกติเท่านั้น)
แต่ละไฟล์คำนวณ feature แยกกัน (sliding window ไม่ข้ามไฟล์) แล้วรวมเป็นชุดเดียว

In [ ]:
sources = []
if USE_SYNTHETIC:
    syn = resolve_path(T["packets_csv"])
    if not syn.exists():                          # ยังไม่เคยสร้าง -> สร้างให้
        from training.generate_data import save_packets_csv
        save_packets_csv(generate(T["n_packets"], SEED), syn)
    sources.append(syn)
sources += normal_csvs
assert sources, "ไม่มีข้อมูลเทรน: วาง pcap ใน training/pcap/normal/ หรือตั้ง USE_SYNTHETIC = True"

from collections import Counter
packets, blocks, summary = [], [], []
for src in sources:
    pk = load_packets_csv(src)
    n0 = len(pk)
    if DROP_ICMP_ECHO:
        pk = [p for p in pk if not (p["proto"] == "ICMP" and p["icmp_type"] in (0, 8))]
    pk.sort(key=lambda p: p["ts"])
    blocks.append(build_features(pk, WINDOW_SEC))
    packets += pk
    c = Counter(p["proto"] for p in pk)
    summary.append({"ไฟล์": Path(src).name, "packets": len(pk), "ตัด ping": n0 - len(pk),
                    "TCP": c["TCP"], "UDP": c["UDP"], "ICMP": c["ICMP"], "อื่นๆ": c["OTHER"]})
X = np.vstack(blocks)
display(Markdown(ev.md_table(summary)))
print("รวม", X.shape[0], "packets x", X.shape[1], "features")

## 3. ดูการกระจายของ feature

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, name in zip(axes.ravel(), ["length", "ttl", "icmp_type", "ctx_pkts",
                                   "ctx_dst_ports", "ctx_dst_ips", "ctx_low_ports", "ctx_ip_ratio"]):
    ax.hist(X[:, FEATURE_NAMES.index(name)], bins=40, color="tab:blue")
    ax.set_title(name); ax.set_yscale("log")
fig.tight_layout(); plt.show()

binary = [n for n in FEATURE_NAMES if n.startswith(("proto_", "flag_", "port_"))]
plt.figure(figsize=(14, 3))
plt.bar(binary, [X[:, FEATURE_NAMES.index(n)].mean() for n in binary])
plt.xticks(rotation=60, ha="right"); plt.ylabel("สัดส่วน packet"); plt.title("feature แบบ 0/1")
plt.tight_layout(); plt.show()

## 4. เทรน StandardScaler + Isolation Forest
เทรนด้วย traffic ปกติอย่างเดียว (unsupervised) — ไม่ต้องใช้ label

In [ ]:
t0 = time.time()
scaler, model = train_model(X, N_ESTIMATORS, SEED)
s_train = anomaly_scores(model, scaler, X)
print(f"เทรนเสร็จใน {time.time() - t0:.1f}s | score ข้อมูลเทรน: median={np.median(s_train):.3f} "
      f"p99={np.quantile(s_train, .99):.3f} max={s_train.max():.3f}")

## 5. ชุดทดสอบที่มี label
- **จำลอง** (`USE_SIM_TEST`): traffic ปกติชุดใหม่ (seed ต่างจากตอนเทรน) + ping + nmap หลายแบบ
- **จาก pcap** ใน `training/pcap/test/`: ติด label ด้วยกฎใน `labels.csv` (packet ที่ไม่ตรงกฎ = ปกติ)

In [ ]:
rules = ev.load_label_rules(LABELS_FILE)
test_blocks = ev.simulated_test_set(4000, SEED + 4200, WINDOW_SEC) if USE_SIM_TEST else []
for csv_path in test_csvs:
    pk = sorted(load_packets_csv(csv_path), key=lambda p: p["ts"])
    names = ev.label_packets(pk, rules)
    n_att = sum(n != ev.NORMAL for n in names)
    print(f"{csv_path.name}: {len(pk):,} packets, ติด label โจมตี {n_att:,}")
    if test_csvs and not rules:
        print("  ⚠️ labels.csv ยังไม่มีกฎ -> ทุก packet ถูกนับเป็นปกติ")
    test_blocks.append((csv_path.name, pk, names))
assert test_blocks, "ไม่มีชุดทดสอบ: ตั้ง USE_SIM_TEST = True หรือวาง pcap ใน training/pcap/test/"

X_test, y_test, names_test = ev.blocks_to_xy(test_blocks, WINDOW_SEC)
s_test = anomaly_scores(model, scaler, X_test)
print(f"ชุดทดสอบ: {len(y_test):,} packets | ปกติ {int((y_test == 0).sum()):,} | โจมตี {int(y_test.sum()):,}")

## 6. เลือก threshold
- **recommended**: กึ่งกลางระหว่าง score ของ traffic ปกติ (quantile `fp_quantile`) กับ ping ต่ำสุด — ไม่ได้ดูชุดทดสอบ (ยุติธรรม)
- **best_f1**: ค่าที่ให้ F1 สูงสุดบนชุดทดสอบ — ⚠️ ได้ตัวเลขสวยเกินจริง เพราะเลือกจากชุดที่ใช้วัดผลเอง

In [ ]:
# threshold แนะนำ: ใช้ traffic ปกติจำลองชุดแยก (validation) + ping จำลอง — ไม่แตะชุดทดสอบ
val = [p for p in generate(4000, SEED + 1) if p["proto"] != "ICMP"]
s_val = anomaly_scores(model, scaler, build_features(val, WINDOW_SEC))
s_ping = np.concatenate([anomaly_scores(model, scaler, build_features(v, WINDOW_SEC))
                         for k, v in make_scenarios().items() if is_ping_scenario(k)])
thr_rec, note = recommend_threshold(s_val, s_ping, FP_QUANTILE)
thr_f1 = ev.best_f1_threshold(y_test, s_test)
THRESHOLD = {"recommended": thr_rec, "best_f1": thr_f1}.get(THRESHOLD_MODE, THRESHOLD_MODE)
THRESHOLD = float(THRESHOLD)
print(f"recommended = {thr_rec:.4f}  ({note})")
print(f"best_f1     = {thr_f1:.4f}")
print(f"ใช้ THRESHOLD = {THRESHOLD:.4f}  (โหมด {THRESHOLD_MODE})")

grid = sorted(set(np.round(np.arange(0.55, 0.76, 0.02), 3).tolist() + [round(thr_rec, 4), round(thr_f1, 4)]))
sweep = ev.threshold_sweep(y_test, s_test, grid)
cols = ["threshold", "TP", "FP", "TN", "FN", "Accuracy", "Precision", "Recall (TPR)", "F1", "FPR"]
display(Markdown("**ผลที่ threshold ต่างๆ** (เลื่อนสูง = alert หลอกน้อยลง แต่จับพลาดมากขึ้น)\n\n" +
                 ev.md_table([{c: r[c] for c in cols} for r in sweep])))

## 7. ผลการวัด — Confusion Matrix และ Metrics

In [ ]:
y_pred = (s_test >= THRESHOLD).astype(int)
m = ev.binary_metrics(y_test, y_pred)
m["ROC-AUC"] = ev.auc(y_test, s_test)

display(Markdown(f"""
### Confusion Matrix (threshold = {THRESHOLD:.4f})
| | ทำนาย: ปกติ | ทำนาย: โจมตี |
|---|---|---|
| **จริง: ปกติ** | TN = {m['TN']:,} | FP = {m['FP']:,} |
| **จริง: โจมตี** | FN = {m['FN']:,} | TP = {m['TP']:,} |

- **TP** จับการโจมตีได้ถูก · **FN** การโจมตีที่หลุด
- **FP** alert หลอก (ปกติแต่ถูกมองว่าแปลก) · **TN** ปกติและตอบว่าปกติ
"""))
display(Markdown("### Metrics\n" + ev.metrics_md(m)))
display(Markdown("### แยกตาม label\n(แถว normal = อัตรา alert หลอก, แถวโจมตี = อัตราที่จับได้)\n\n" +
                 ev.md_table(ev.per_label_table(names_test, y_pred))))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ev.plot_confusion(m, axes[0], f"Confusion Matrix (thr={THRESHOLD:.3f})")
ev.plot_roc(y_test, s_test, axes[1])
ev.plot_score_hist(y_test, s_test, THRESHOLD, axes[2])
fig.tight_layout(); plt.show()

> **อ่านผลอย่างไร:** dataset IDS มี traffic ปกติเยอะกว่าการโจมตีมาก **Accuracy อย่างเดียวหลอกตา**
> ให้ดู **Recall** (จับการโจมตีได้กี่ %), **FPR** (alert หลอกกี่ %) และ **F1** ควบคู่
> ส่วน nmap จะหลุดช่วงแรกๆ (FN) เป็นเรื่องปกติ เพราะต้องรอให้ window 5 วินาทีเห็นพฤติกรรมก่อน

## 8. บันทึกโมเดล
บันทึกลง `system/models/` → detection-service โหลดใช้ทันทีเมื่อรีสตาร์ท (`detection.threshold: auto` ใน config จะใช้ threshold นี้)

In [ ]:
extra = {
    "threshold_mode": str(THRESHOLD_MODE),
    "test_metrics": {k: (round(v, 5) if isinstance(v, float) else v) for k, v in m.items()},
    "test_packets": int(len(y_test)),
}
src_names = ", ".join(Path(s).name for s in sources)
paths = save_artifacts(model, scaler, THRESHOLD, len(X), src_names, cfg, extra)
for k, p in paths.items():
    print(f"{k:12} -> {p}")
if str(cfg["detection"].get("threshold", "auto")).lower() != "auto":
    print("⚠️ config detection.threshold เป็นตัวเลข -> detection-service จะใช้ค่าใน config แทนค่าที่บันทึกนี้")
print("\nขั้นต่อไป: ปิด run_all.py เดิม (Ctrl+C) แล้วรันใหม่ เพื่อโหลดโมเดลใหม่")